# Telegram Save Restricted Content Bot - Colab Runner

**Qadamlar:**
1. Loyihani yuklash (GitHub yoki ZIP upload)
2. Kutubxonalarni o'rnatish
3. Sozlamalarni kiritish (Secrets yoki qo'lda)
4. Botni ishga tushirish

**To'xtatish:** `Runtime > Interrupt execution`

---
## 1. Loyihani yuklash

In [ ]:
# ============================================================
# LOYIHA MANBASI - birini tanlang
# ============================================================
PROJECT_SOURCE = "github"  # "github" yoki "upload"

# GitHub sozlamalari
GITHUB_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # <-- O'ZGARTIRING
BRANCH = "main"
GITHUB_TOKEN = ""  # Shaxsiy repo uchun: ghp_xxxx

# ============================================================
import os, sys, shutil, glob

PROJECT_DIR = "/content/project"

def find_project_root(base):
    """config.py + core/ papkasi bor joyni topadi - necha qatlam ichida bo'lmasin."""
    if os.path.isfile(os.path.join(base, "config.py")) and os.path.isdir(os.path.join(base, "core")):
        return base
    for root, dirs, files in os.walk(base):
        # venv, __pycache__, .git papkalarini o'tkazib yuborish
        dirs[:] = [d for d in dirs if d not in ("venv", "__pycache__", ".git", "node_modules")]
        if "config.py" in files and "core" in dirs:
            return root
    return None

if PROJECT_SOURCE == "github":
    if os.path.isfile(os.path.join(PROJECT_DIR, "config.py")):
        print(f"Loyiha allaqachon mavjud: {PROJECT_DIR}")
    else:
        if os.path.exists(PROJECT_DIR):
            shutil.rmtree(PROJECT_DIR)
        url = GITHUB_URL
        if GITHUB_TOKEN:
            url = GITHUB_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
        ret = os.system(f"git clone --branch {BRANCH} --depth 1 {url} {PROJECT_DIR}")
        if ret != 0:
            raise RuntimeError("Git clone xatosi! GITHUB_URL va GITHUB_TOKEN ni tekshiring.")
        # Agar repo ichida papka ichida loyiha bo'lsa
        real = find_project_root(PROJECT_DIR)
        if real and real != PROJECT_DIR:
            tmp = PROJECT_DIR + "_tmp"
            shutil.move(real, tmp)
            shutil.rmtree(PROJECT_DIR)
            shutil.move(tmp, PROJECT_DIR)
        print("GitHub'dan klonlandi")

elif PROJECT_SOURCE == "upload":
    from google.colab import files as colab_files
    print("ZIP yoki TAR arxivni yuklang:")
    uploaded = colab_files.upload()
    if not uploaded:
        raise RuntimeError("Hech narsa yuklanmadi!")
    archive_name = list(uploaded.keys())[0]
    print(f"Yuklandi: {archive_name}")

    extract_tmp = "/content/_extract_tmp"
    if os.path.exists(extract_tmp):
        shutil.rmtree(extract_tmp)
    os.makedirs(extract_tmp)

    if archive_name.endswith(".zip"):
        import zipfile
        with zipfile.ZipFile(archive_name, "r") as zf:
            zf.extractall(extract_tmp)
    elif archive_name.endswith((".tar.gz", ".tgz", ".tar", ".tar.bz2")):
        import tarfile
        with tarfile.open(archive_name, "r:*") as tf:
            tf.extractall(extract_tmp)
    else:
        raise RuntimeError(f"Qo'llab-quvvatlanmaydigan format: {archive_name}")

    real = find_project_root(extract_tmp)
    if real is None:
        # Ro'yxat ko'rsatish - debug uchun
        for r, d, f in os.walk(extract_tmp):
            level = r.replace(extract_tmp, "").count(os.sep)
            if level < 3:
                print(f"  {'  '*level}{os.path.basename(r)}/")
                for fn in f[:5]:
                    print(f"  {'  '*(level+1)}{fn}")
        raise RuntimeError("Arxiv ichidan config.py + core/ topilmadi! Arxiv tuzilishini tekshiring.")

    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    shutil.copytree(real, PROJECT_DIR)
    shutil.rmtree(extract_tmp)
    try:
        os.remove(archive_name)
    except OSError:
        pass
    print(f"Arxivdan ochildi")

else:
    raise ValueError(f"Noto'g'ri PROJECT_SOURCE: {PROJECT_SOURCE}")

# Yakuniy tekshirish
assert os.path.isfile(os.path.join(PROJECT_DIR, "config.py")), \
    f"config.py topilmadi! Loyiha tuzilishini tekshiring."
assert os.path.isdir(os.path.join(PROJECT_DIR, "core")), \
    f"core/ papkasi topilmadi!"
print(f"Loyiha tayyor: {PROJECT_DIR}")

---
## 2. Kutubxonalarni o'rnatish

In [ ]:
import subprocess, sys, os

PROJECT_DIR = "/content/project"
req_file = os.path.join(PROJECT_DIR, "requirements.txt")
assert os.path.exists(req_file), "requirements.txt topilmadi!"

print("Kutubxonalar o'rnatilmoqda (bu 1-2 daqiqa olishi mumkin)...")
ret = subprocess.call([
    sys.executable, "-m", "pip", "install",
    "--no-cache-dir", "--prefer-binary", "--progress-bar", "off",
    "-r", req_file
])
if ret != 0:
    print("OGOHLANTIRISH: Ba'zi kutubxonalar o'rnatilmagan bo'lishi mumkin.")

# uvloop tekshiruvi (requirements.txt ichida sys_platform != win32 bilan)
try:
    import uvloop
    print(f"uvloop: {uvloop.__version__}")
except ImportError:
    print("uvloop: o'rnatilmagan (Linux bo'lmasa normal)")

# nest_asyncio (requirements.txt ichida)
try:
    import nest_asyncio
    print("nest_asyncio: OK")
except ImportError:
    subprocess.call([sys.executable, "-m", "pip", "install", "-q", "nest_asyncio"])

print(f"\nPython: {sys.version}")
print("O'rnatish tugadi.")

---
## 3. Sozlamalar

**Xavfsiz usul (tavsiya):** Colab Secrets:
1. Chap panelda **kalit ikonkasini** bosing
2. Quyidagi nomlar bilan secret qo'shing: `BOT_TOKEN`, `API_ID`, `API_HASH`, `OWNER_ID`, `DB_URI`
3. Har biri uchun **Notebook access** ni yoqing

**Oddiy usul:** Quyidagi maydonlarga qo'lda yozing.

In [ ]:
import os

PROJECT_DIR = "/content/project"
_keys = ["BOT_TOKEN", "API_ID", "API_HASH", "OWNER_ID", "DB_URI", "OWNER_USERNAME"]
_loaded = set()

# ============================================================
# 1-MANBA: .env fayl (ZIP bilan yuklangan loyiha ichida)
# ============================================================
_env_path = os.path.join(PROJECT_DIR, ".env")
if os.path.isfile(_env_path):
    print(f".env fayl topildi: {_env_path}")
    with open(_env_path) as _f:
        for _line in _f:
            _line = _line.strip()
            if not _line or _line.startswith("#") or "=" not in _line:
                continue
            _k, _v = _line.split("=", 1)
            _k, _v = _k.strip(), _v.strip()
            if _k in _keys and _v:
                os.environ[_k] = _v
                _loaded.add(_k)
    if _loaded:
        print(f".env dan yuklandi: {', '.join(sorted(_loaded))}")

# ============================================================
# 2-MANBA: Colab Secrets (qolgan kalitlar uchun)
# ============================================================
try:
    from google.colab import userdata
    for _k in _keys:
        if _k in _loaded:
            continue
        try:
            _v = userdata.get(_k)
            if _v:
                os.environ[_k] = str(_v)
                _loaded.add(_k)
        except Exception:
            pass
    _secrets = _loaded - set()  # har qanday yangi yuklangan
    if _secrets:
        print(f"Colab Secrets dan yuklandi: {', '.join(sorted(_loaded))}")
except Exception:
    pass

# ============================================================
# 3-MANBA: Qo'lda kiritish (hali bo'sh qolganlar uchun)
# ============================================================
if "BOT_TOKEN" not in _loaded and not os.environ.get("BOT_TOKEN"):
    os.environ["BOT_TOKEN"]   = ""  # <-- BotFather dan olingan token
if "API_ID" not in _loaded and not os.environ.get("API_ID"):
    os.environ["API_ID"]      = ""  # <-- my.telegram.org dan (raqam)
if "API_HASH" not in _loaded and not os.environ.get("API_HASH"):
    os.environ["API_HASH"]    = ""  # <-- my.telegram.org dan
if "OWNER_ID" not in _loaded and not os.environ.get("OWNER_ID"):
    os.environ["OWNER_ID"]    = ""  # <-- Sizning Telegram ID
if "DB_URI" not in _loaded and not os.environ.get("DB_URI"):
    os.environ["DB_URI"]      = ""  # <-- MongoDB connection string

# ============================================================
# Tekshirish
# ============================================================
_ok = True
for _k in ["BOT_TOKEN", "API_ID", "API_HASH"]:
    _v = os.environ.get(_k, "")
    if not _v or _v == "0":
        print(f"XATO: {_k} bo'sh! .env ga yozing, Secrets ga qo'shing, yoki yuqorida to'ldiring.")
        _ok = False
    else:
        print(f"{_k}: ...{_v[-6:]}")

for _k in ["OWNER_ID", "DB_URI"]:
    _v = os.environ.get(_k, "")
    if _v and _v != "0":
        print(f"{_k}: ...{_v[-6:]}")
    else:
        print(f"{_k}: (kiritilmagan - ixtiyoriy)")

if not _ok:
    raise SystemExit("Majburiy sozlamalar to'ldirilmagan!")
print("\nSozlamalar tayyor.")

---
## 4. Botni ishga tushirish

To'xtatish: **Runtime > Interrupt execution**

In [ ]:
import os, sys, asyncio, logging, time

# ============================================================
# Muhitni sozlash
# ============================================================
PROJECT_DIR = "/content/project"
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

for d in ["sessions", "downloads/temp", "logs", "data"]:
    os.makedirs(os.path.join(PROJECT_DIR, d), exist_ok=True)

print(f"CWD: {os.getcwd()}")
print(f"sessions/ mavjud: {os.path.isdir('sessions')}")
print(f"data/ mavjud: {os.path.isdir('data')}")

# ============================================================
# Asyncio patch - Colab/Jupyter da loop allaqachon ishlaydi
# ============================================================
import nest_asyncio
nest_asyncio.apply()

from core.compat import configure_event_loop_policy
configure_event_loop_policy()

# ============================================================
# Cloud-optimized logging
# ============================================================
logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
for _n in ["pymongo", "pyrogram", "pyrogram.session",
           "pyrogram.connection", "pyrogram.dispatcher"]:
    logging.getLogger(_n).setLevel(logging.ERROR)
logging.getLogger("TechVJ").setLevel(logging.INFO)
logging.getLogger("core").setLevel(logging.INFO)

# ============================================================
# Cloud runtime tuning (manba fayllar o'zgartirilMAYDI)
# ============================================================
try:
    from core.downloader import worker as _w
    _w.DEFAULT_WORKER_COUNT = 12
    _w.MAX_WORKER_COUNT = 64
    _w.MIN_WORKER_COUNT = 4
except Exception:
    pass

try:
    from core.downloader import adaptive_engine as _ae
    _ae.NETWORK_CHUNK_SIZE = 512 * 1024
    _ae.CHECKPOINT_BYTES_MEDIUM = 10 * 1024 * 1024
    _ae.CHECKPOINT_BYTES_LARGE = 50 * 1024 * 1024
except Exception:
    pass

# ============================================================
# Bot
# ============================================================
from main import Bot
from pyrogram import idle

bot = Bot()

async def run_bot():
    t0 = time.time()
    try:
        await bot.start()
        print("=" * 50)
        print(f"Bot ishga tushdi! ({time.time()-t0:.1f}s)")
        print("To'xtatish: Runtime > Interrupt execution")
        print("=" * 50)
        await idle()
    except (KeyboardInterrupt, asyncio.CancelledError):
        print("\nBot to'xtatilmoqda...")
    finally:
        try:
            await bot.stop()
        except Exception:
            pass
        print("Bot to'xtadi.")

loop = asyncio.get_event_loop()
try:
    loop.run_until_complete(run_bot())
except KeyboardInterrupt:
    print("\nFoydalanuvchi to'xtatdi.")
except Exception as e:
    print(f"Xato: {e}")